# Hydro-Art Master Poster — Pacific County + Texas

A single **48x72"** museum-scale poster combining every key visualization from
both the Pacific County (WA) and Texas comprehensive watershed reports. 40+ panels
across 7 rivers, 2 states, 60-85 year gauge records, and ENSO/PDO teleconnections.


In [1]:
# ── imports ──────────────────────────────────────────────────────────────
import sys
from pathlib import Path

REPO = Path(".").resolve().parent
sys.path.insert(0, str(REPO))

import numpy as np
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import gaussian_kde
from PIL import Image

from src import flow_metrics as fm
from tools.nwis_gauge import GaugeProvider
from tools.climate_index import ClimateIndexProvider
from tools.monthly_flow import MONTH_ABBR

NAS = "/Volumes/home/data/hydro-art"
OUT = REPO / "output" / "prints"
OUT.mkdir(parents=True, exist_ok=True)

%matplotlib inline

# ── theme ──
BG = "#07080c"; PANEL = "#10121b"; EDGE = "#232838"
TEXT = "#e6ebf5"; TICK = "#8891a8"
CYAN = "#00ffff"; MAGENTA = "#ff4d9a"; LIME = "#00ff9c"
VIOLET = "#9d00ff"; AMBER = "#ff9c3a"; RED = "#ff4d5e"
COOL_BLUE = "#00e5ff"; MUTED = "#8891a8"

plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": PANEL,
    "axes.edgecolor": EDGE, "axes.labelcolor": TEXT,
    "text.color": TEXT, "xtick.color": TICK, "ytick.color": TICK,
    "figure.dpi": 100, "font.size": 9,
})

def complete_years(obs):
    return {y: v for y, v in obs.items() if np.all(np.isfinite(v))}

def sty(ax):
    ax.set_facecolor(PANEL)
    for s in ax.spines.values(): s.set_color(EDGE)
    ax.tick_params(colors=TICK, labelsize=6)

print("ready")


ready


In [2]:
# ── load all data ────────────────────────────────────────────────────────
# Pacific County
willapa = GaugeProvider("12013500", NAS)
willapa_obs = willapa.monthly_means(1961, 2024)
naselle = GaugeProvider("12010000", NAS)
naselle_obs = naselle.monthly_means(1960, 2003)

# Texas
GAUGES_TX = {
    "Brazos":    ("08096500", 1960, 2024),
    "Colorado":  ("08158000", 1960, 2024),
    "Trinity":   ("08066500", 1960, 2024),
    "Guadalupe": ("08176500", 1940, 2024),
    "Sabine":    ("08030500", 1940, 2024),
}
tx = {}
for name, (site, s, e) in GAUGES_TX.items():
    g = GaugeProvider(site, NAS)
    tx[name] = g.monthly_means(s, e)

# Climate
oni = ClimateIndexProvider("oni", NAS).index_by_year(1940, 2024)
pdo = ClimateIndexProvider("pdo", NAS).index_by_year(1940, 2024)

# Pre-rendered assets
pacific_map = REPO / "output" / "washington-pacific.png"
tx_hero = REPO / "output" / "year_texas" / "week_10.png"
tx_frames = sorted((REPO / "output" / "year_texas").glob("week_*.png"))
tx_frames = [f for f in tx_frames if "_frame" not in f.name]

RIVER_COLORS = {
    "Willapa": CYAN, "Naselle": MAGENTA,
    "Brazos": CYAN, "Colorado": MAGENTA, "Trinity": LIME,
    "Guadalupe": AMBER, "Sabine": VIOLET,
}

print(f"Willapa: {len(complete_years(willapa_obs))} yr")
print(f"Naselle: {len(complete_years(naselle_obs))} yr")
for n in tx: print(f"{n}: {len(complete_years(tx[n]))} yr")
print(f"TX animation frames: {len(tx_frames)}")


Willapa: 63 yr
Naselle: 43 yr
Brazos: 65 yr
Colorado: 65 yr
Trinity: 65 yr
Guadalupe: 85 yr
Sabine: 85 yr
TX animation frames: 26


---
## Compose the Master Poster

**Layout:** 48x72" at 150 DPI = 7200x10800 px. Grid: 8 columns x 12 rows.

- Row 0: Title banner
- Rows 1-2: Hero maps (Pacific County left, Texas right)
- Rows 3-4: Pacific County panels (typical year, trend, FDC, composites, anomaly, heatmap)
- Rows 5-6: Texas panels (same set for 5 rivers)
- Rows 7-8: Cross-analysis (all-rivers radar, correlation matrix, climate indices, ENSO)
- Rows 9-10: Forecasts + record books
- Row 11: Animation frames + attribution


In [3]:
m = range(1, 13)
neon_cmap = LinearSegmentedColormap.from_list(
    "neon", ["#07080c", "#0a1628", "#002b4d", "#005577", "#00aaaa",
             "#00ffcc", "#66ff99", "#ffff66", "#ff9933", "#ff3366"])
cmap_div = LinearSegmentedColormap.from_list("anom", [COOL_BLUE, "#10121b", RED])

fig = plt.figure(figsize=(48, 72), facecolor=BG, dpi=150)
gs = GridSpec(12, 8, figure=fig, hspace=0.22, wspace=0.18,
              left=0.025, right=0.975, top=0.97, bottom=0.008,
              height_ratios=[0.3, 1.8, 1.8, 1, 1, 1, 1, 1, 1, 1, 1, 0.6])

# ════════════════════════════════════════════════════════════════════════
# ROW 0: TITLE BANNER
# ════════════════════════════════════════════════════════════════════════
ax = fig.add_subplot(gs[0, :])
ax.set_facecolor(BG); ax.set_axis_off()
ax.text(0.5, 0.75, "AMERICAN RIVERS", ha="center", va="top",
        fontsize=64, fontweight="bold", color=CYAN, fontfamily="monospace",
        transform=ax.transAxes)
ax.text(0.5, 0.15,
        "PACIFIC COUNTY, WA  +  TEXAS  |  7 RIVERS  |  60-85 YEAR GAUGE RECORDS  |  USGS NHDPlus HR  |  NOAA ONI + PDO",
        ha="center", va="bottom", fontsize=16, color=TICK, fontfamily="monospace",
        transform=ax.transAxes)

# ════════════════════════════════════════════════════════════════════════
# ROWS 1-2: HERO MAPS
# ════════════════════════════════════════════════════════════════════════
# Pacific County map (left half)
ax = fig.add_subplot(gs[1:3, :4])
ax.set_facecolor(BG)
for s in ax.spines.values(): s.set_visible(False)
if pacific_map.exists():
    ax.imshow(plt.imread(pacific_map))
ax.set_axis_off()
ax.set_title("PACIFIC COUNTY, WA  |  89,083 RIVER SEGMENTS", fontsize=14,
             color=CYAN, pad=10, fontfamily="monospace")

# Texas map (right half)
ax = fig.add_subplot(gs[1:3, 4:])
ax.set_facecolor(BG)
for s in ax.spines.values(): s.set_visible(False)
if tx_hero.exists():
    ax.imshow(plt.imread(tx_hero))
ax.set_axis_off()
ax.set_title("TEXAS  |  21 HUC4 BASINS  |  ELEVATION TINT  |  PEAK FLOW", fontsize=14,
             color=CYAN, pad=10, fontfamily="monospace")

# ════════════════════════════════════════════════════════════════════════
# ROWS 3-4: PACIFIC COUNTY PANELS
# ════════════════════════════════════════════════════════════════════════

# --- 3,0-1: Willapa Typical Year ---
ax = fig.add_subplot(gs[3, 0:2]); sty(ax)
for obs_data, name, color in [(willapa_obs, "Willapa", CYAN), (naselle_obs, "Naselle", MAGENTA)]:
    years = sorted(obs_data.keys())
    stack = np.array([obs_data[y] for y in years])
    mean = np.nanmean(stack, axis=0)
    p10, p90 = np.nanpercentile(stack, 10, axis=0), np.nanpercentile(stack, 90, axis=0)
    ax.fill_between(m, p10, p90, color=color, alpha=0.08)
    ax.plot(m, mean, color=color, lw=2, label=name)
ax.set_xticks(list(m)); ax.set_xticklabels(MONTH_ABBR, fontsize=5)
ax.set_ylabel("cfs", fontsize=7); ax.set_title("Pacific Co — Typical Year", fontsize=9, pad=4)
ax.legend(fontsize=6, frameon=False)

# --- 3,2-3: Long Record Trend ---
ax = fig.add_subplot(gs[3, 2:4]); sty(ax)
for obs_data, name, color in [(willapa_obs, "Willapa", CYAN), (naselle_obs, "Naselle", MAGENTA)]:
    clean = complete_years(obs_data)
    years = sorted(clean.keys())
    annual = np.array([float(np.mean(clean[y])) for y in years])
    mk = fm.mann_kendall(annual)
    slope = fm.sens_slope(annual)
    x = np.arange(len(years))
    fit = np.median(annual) + slope * (x - np.median(x))
    ax.plot(years, annual, "o-", color=color, lw=0.8, ms=2, alpha=0.7, label=f"{name} {mk.trend}")
    ax.plot(years, fit, "--", color=color, lw=1, alpha=0.5)
ax.set_ylabel("annual mean (cfs)", fontsize=7)
ax.set_title("Pacific Co — Long Record Trend", fontsize=9, pad=4)
ax.legend(fontsize=6, frameon=False)

# --- 3,4-5: FDC by Decade (Willapa) ---
ax = fig.add_subplot(gs[3, 4:6]); sty(ax)
clean_w = complete_years(willapa_obs)
quantiles = [0, 5, 10, 25, 50, 75, 90, 95, 100]
decades = fm.decade_flow_duration(clean_w, quantiles)
cmap_v = plt.get_cmap("viridis")
span = max(1, len(decades) - 1)
for j, d in enumerate(decades):
    ax.plot(d.quantiles, np.clip(d.flows, 1e-2, None), "-o", ms=2,
            color=cmap_v(j / span), lw=1.2, label=f"{d.decade}s")
ax.set_yscale("log"); ax.set_xlabel("exceedance %", fontsize=7)
ax.set_ylabel("cfs", fontsize=7); ax.set_title("Willapa — FDC by Decade", fontsize=9, pad=4)
ax.legend(fontsize=5, frameon=False, ncol=2)

# --- 3,6-7: ENSO Composites (Willapa) ---
ax = fig.add_subplot(gs[3, 6:]); sty(ax)
pc = fm.composite_hydrographs(clean_w, oni)
for label, vec, color in [("El Nino", pc.warm, RED), ("Neutral", pc.neutral, MUTED), ("La Nina", pc.cool, COOL_BLUE)]:
    if vec is not None:
        ax.plot(m, vec, "-o", ms=3, lw=1.8, color=color, label=label)
ax.set_xticks(list(m)); ax.set_xticklabels(MONTH_ABBR, fontsize=5)
ax.set_ylabel("cfs", fontsize=7); ax.set_title("Willapa — ENSO Composites", fontsize=9, pad=4)
ax.legend(fontsize=6, frameon=False)

# --- Row 4: Anomaly stripes, Heatmap, Box, Forecast ---
# 4,0-1: Anomaly stripes (both rivers stacked)
ax1 = fig.add_subplot(gs[4, 0:2]); sty(ax1)
for obs_data, name, yoff in [(willapa_obs, "Willapa", 1), (naselle_obs, "Naselle", 0)]:
    years = sorted(obs_data.keys())
    annual = np.array([float(np.nanmean(obs_data[y])) for y in years])
    normal = np.mean(annual)
    anomalies = annual - normal
    vmax = max(abs(anomalies.min()), abs(anomalies.max()))
    for y_val, a in zip(years, anomalies):
        color = cmap_div((a / vmax + 1) / 2)
        ax1.bar(y_val, 0.4, bottom=yoff * 0.5, color=color, width=1.0, edgecolor="none")
ax1.set_yticks([0.2, 0.7]); ax1.set_yticklabels(["Naselle", "Willapa"], fontsize=6)
ax1.set_title("Pacific Co — Anomaly Stripes", fontsize=9, pad=4)

# 4,2-3: Heatmap (Willapa)
ax = fig.add_subplot(gs[4, 2:4]); sty(ax)
years_w = sorted(willapa_obs.keys())
mat = np.array([willapa_obs[y] for y in years_w])
im = ax.imshow(mat, aspect="auto", cmap=neon_cmap, interpolation="nearest")
ax.set_xticks(range(12)); ax.set_xticklabels(MONTH_ABBR, fontsize=5)
ytick_idx = list(range(0, len(years_w), 10))
ax.set_yticks(ytick_idx); ax.set_yticklabels([years_w[i] for i in ytick_idx], fontsize=5)
ax.set_title("Willapa — Flow Heatmap", fontsize=9, pad=4)
fig.colorbar(im, ax=ax, label="cfs", shrink=0.7, pad=0.02)

# 4,4-5: Monthly box plots (Willapa)
ax = fig.add_subplot(gs[4, 4:6]); sty(ax)
data = [[willapa_obs[y][mi] for y in years_w if np.isfinite(willapa_obs[y][mi])] for mi in range(12)]
ax.boxplot(data, patch_artist=True, tick_labels=MONTH_ABBR,
           boxprops=dict(facecolor=CYAN, alpha=0.3), medianprops=dict(color=AMBER, lw=2),
           whiskerprops=dict(color=MUTED), capprops=dict(color=MUTED),
           flierprops=dict(marker="o", markerfacecolor=RED, ms=2, alpha=0.5))
ax.set_ylabel("cfs", fontsize=7); ax.set_title("Willapa — Monthly Distributions", fontsize=9, pad=4)

# 4,6-7: Climatological Forecast (Willapa)
ax = fig.add_subplot(gs[4, 6:]); sty(ax)
latest_w = max(years_w)
clim_yrs = [y for y in years_w if y >= latest_w - 29]
clim_stack = np.array([willapa_obs[y] for y in clim_yrs])
clim_mean = np.nanmean(clim_stack, axis=0)
cp10, cp90 = np.nanpercentile(clim_stack, 10, axis=0), np.nanpercentile(clim_stack, 90, axis=0)
ax.fill_between(m, cp10, cp90, color=VIOLET, alpha=0.15)
ax.plot(m, clim_mean, color=CYAN, lw=2.5, label="30-yr clim")
ax.plot(m, willapa_obs[latest_w], "o-", color=AMBER, lw=2, ms=3, label=f"obs {latest_w}")
ax.set_xticks(list(m)); ax.set_xticklabels(MONTH_ABBR, fontsize=5)
ax.set_ylabel("cfs", fontsize=7); ax.set_title(f"Willapa — Forecast {latest_w+1}", fontsize=9, pad=4)
ax.legend(fontsize=6, frameon=False)

# ════════════════════════════════════════════════════════════════════════
# ROWS 5-6: TEXAS PANELS
# ════════════════════════════════════════════════════════════════════════

# --- 5,0-1: Texas Typical Year (all 5 rivers normalized) ---
ax = fig.add_subplot(gs[5, 0:2]); sty(ax)
for name, obs_data in tx.items():
    clean = complete_years(obs_data)
    years = sorted(clean.keys())
    stack = np.array([clean[y] for y in years])
    mean = np.nanmean(stack, axis=0)
    norm = (mean - mean.min()) / (mean.max() - mean.min() + 1e-9)
    ax.plot(m, norm, "o-", color=RIVER_COLORS[name], lw=1.8, ms=3, label=name)
ax.set_xticks(list(m)); ax.set_xticklabels(MONTH_ABBR, fontsize=5)
ax.set_ylabel("normalized", fontsize=7)
ax.set_title("Texas — Seasonal Shape (normalized)", fontsize=9, pad=4)
ax.legend(fontsize=5, frameon=False, ncol=2)

# --- 5,2-3: Texas Long Record (all 5) ---
ax = fig.add_subplot(gs[5, 2:4]); sty(ax)
for name, obs_data in tx.items():
    clean = complete_years(obs_data)
    years = sorted(clean.keys())
    annual = np.array([float(np.mean(clean[y])) for y in years])
    ax.plot(years, annual, "-", color=RIVER_COLORS[name], lw=1, alpha=0.7, label=name)
ax.set_ylabel("annual mean (cfs)", fontsize=7)
ax.set_title("Texas — Long Record (5 rivers)", fontsize=9, pad=4)
ax.legend(fontsize=5, frameon=False, ncol=2)

# --- 5,4-5: Texas FDC (Brazos) ---
ax = fig.add_subplot(gs[5, 4:6]); sty(ax)
clean_b = complete_years(tx["Brazos"])
decades_b = fm.decade_flow_duration(clean_b, quantiles)
span_b = max(1, len(decades_b) - 1)
for j, d in enumerate(decades_b):
    ax.plot(d.quantiles, np.clip(d.flows, 1e-2, None), "-o", ms=2,
            color=cmap_v(j / span_b), lw=1.2, label=f"{d.decade}s")
ax.set_yscale("log"); ax.set_xlabel("exceedance %", fontsize=7)
ax.set_ylabel("cfs", fontsize=7); ax.set_title("Brazos — FDC by Decade", fontsize=9, pad=4)
ax.legend(fontsize=4, frameon=False, ncol=2)

# --- 5,6-7: Texas ENSO Composites (Brazos) ---
ax = fig.add_subplot(gs[5, 6:]); sty(ax)
pc_b = fm.composite_hydrographs(clean_b, oni)
for label, vec, color in [("El Nino", pc_b.warm, RED), ("Neutral", pc_b.neutral, MUTED), ("La Nina", pc_b.cool, COOL_BLUE)]:
    if vec is not None:
        ax.plot(m, vec, "-o", ms=3, lw=1.8, color=color, label=label)
ax.set_xticks(list(m)); ax.set_xticklabels(MONTH_ABBR, fontsize=5)
ax.set_ylabel("cfs", fontsize=7); ax.set_title("Brazos — ENSO Composites", fontsize=9, pad=4)
ax.legend(fontsize=6, frameon=False)

# --- Row 6: Texas anomaly, heatmap, box, forecast ---
# 6,0-1: Texas anomaly stripes (5 rivers stacked)
ax = fig.add_subplot(gs[6, 0:2]); sty(ax)
for ri, (name, obs_data) in enumerate(tx.items()):
    years = sorted(obs_data.keys())
    annual = np.array([float(np.nanmean(obs_data[y])) for y in years])
    normal = np.mean(annual)
    anomalies = annual - normal
    vmax = max(abs(anomalies.min()), abs(anomalies.max()))
    for y_val, a in zip(years, anomalies):
        color = cmap_div((a / vmax + 1) / 2)
        ax.bar(y_val, 0.16, bottom=ri * 0.19, color=color, width=1.0, edgecolor="none")
ax.set_yticks([i * 0.19 + 0.08 for i in range(5)])
ax.set_yticklabels(list(tx.keys()), fontsize=5)
ax.set_title("Texas — Anomaly Stripes", fontsize=9, pad=4)

# 6,2-3: Heatmap (Brazos)
ax = fig.add_subplot(gs[6, 2:4]); sty(ax)
years_b = sorted(tx["Brazos"].keys())
mat_b = np.array([tx["Brazos"][y] for y in years_b])
im = ax.imshow(mat_b, aspect="auto", cmap=neon_cmap, interpolation="nearest")
ax.set_xticks(range(12)); ax.set_xticklabels(MONTH_ABBR, fontsize=5)
ytick_idx = list(range(0, len(years_b), 10))
ax.set_yticks(ytick_idx); ax.set_yticklabels([years_b[i] for i in ytick_idx], fontsize=5)
ax.set_title("Brazos — Flow Heatmap", fontsize=9, pad=4)
fig.colorbar(im, ax=ax, label="cfs", shrink=0.7, pad=0.02)

# 6,4-5: Texas box plots (Brazos)
ax = fig.add_subplot(gs[6, 4:6]); sty(ax)
data_b = [[tx["Brazos"][y][mi] for y in years_b if np.isfinite(tx["Brazos"][y][mi])] for mi in range(12)]
ax.boxplot(data_b, patch_artist=True, tick_labels=MONTH_ABBR,
           boxprops=dict(facecolor=CYAN, alpha=0.3), medianprops=dict(color=AMBER, lw=2),
           whiskerprops=dict(color=MUTED), capprops=dict(color=MUTED),
           flierprops=dict(marker="o", markerfacecolor=RED, ms=2, alpha=0.5))
ax.set_ylabel("cfs", fontsize=7); ax.set_title("Brazos — Monthly Distributions", fontsize=9, pad=4)

# 6,6-7: Texas Forecast (Brazos)
ax = fig.add_subplot(gs[6, 6:]); sty(ax)
latest_b = max(years_b)
clim_yrs_b = [y for y in years_b if y >= latest_b - 29]
clim_stack_b = np.array([tx["Brazos"][y] for y in clim_yrs_b])
clim_mean_b = np.nanmean(clim_stack_b, axis=0)
cp10_b, cp90_b = np.nanpercentile(clim_stack_b, 10, axis=0), np.nanpercentile(clim_stack_b, 90, axis=0)
ax.fill_between(m, cp10_b, cp90_b, color=VIOLET, alpha=0.15)
ax.plot(m, clim_mean_b, color=CYAN, lw=2.5, label="30-yr clim")
ax.plot(m, tx["Brazos"][latest_b], "o-", color=AMBER, lw=2, ms=3, label=f"obs {latest_b}")
ax.set_xticks(list(m)); ax.set_xticklabels(MONTH_ABBR, fontsize=5)
ax.set_ylabel("cfs", fontsize=7); ax.set_title(f"Brazos — Forecast {latest_b+1}", fontsize=9, pad=4)
ax.legend(fontsize=6, frameon=False)

# ════════════════════════════════════════════════════════════════════════
# ROWS 7-8: CROSS-ANALYSIS
# ════════════════════════════════════════════════════════════════════════

# 7,0-1: Multi-metric radar (all 7 rivers)
def compute_metrics(obs):
    clean = complete_years(obs)
    years = sorted(clean.keys())
    annual = [float(np.mean(clean[y])) for y in years]
    peaks = [float(np.max(clean[y])) for y in years]
    lows = [float(np.min(clean[y][5:8])) for y in years]
    flash = [fm.flashiness(clean[y]) for y in years]
    mean_flow = np.mean([clean[y] for y in years], axis=0)
    return {
        "Mean": np.mean(annual), "Peak": np.mean(peaks),
        "Low": np.mean(lows), "Flash": np.mean(flash),
        "COT": fm.center_of_timing(mean_flow),
        "Ratio": np.mean([fm.seasonal_ratio(clean[y]) for y in years]),
    }

all_rivers = {"Willapa": willapa_obs, "Naselle": naselle_obs}
all_rivers.update(tx)
all_m = {n: compute_metrics(o) for n, o in all_rivers.items()}
categories = list(list(all_m.values())[0].keys())
N = len(categories)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist() + [0]
all_vals = {k: [all_m[n][k] for n in all_m] for k in categories}

ax = fig.add_subplot(gs[7:9, 0:2], polar=True)
ax.set_facecolor(PANEL)
for name in all_rivers:
    vals = [(all_m[name][k] - min(all_vals[k])) / (max(all_vals[k]) - min(all_vals[k]) + 1e-9)
            for k in categories] + [(all_m[name][categories[0]] - min(all_vals[categories[0]])) / (max(all_vals[categories[0]]) - min(all_vals[categories[0]]) + 1e-9)]
    ax.plot(angles, vals, "o-", color=RIVER_COLORS[name], lw=1.5, ms=3, label=name)
    ax.fill(angles, vals, color=RIVER_COLORS[name], alpha=0.03)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(categories, fontsize=7, color=TEXT)
ax.set_title("All 7 Rivers — Metric Radar", fontsize=10, pad=15)
ax.legend(loc="upper right", fontsize=5, frameon=False)

# 7-8,2-3: Cross-gauge correlation matrix (all 7)
ax = fig.add_subplot(gs[7:9, 2:4]); sty(ax)
rnames = list(all_rivers.keys())
all_years_sets = [set(complete_years(all_rivers[n]).keys()) for n in rnames]
common = sorted(set.intersection(*all_years_sets))
ann_all = {}
for n in rnames:
    clean = complete_years(all_rivers[n])
    ann_all[n] = np.array([float(np.mean(clean[y])) for y in common])
mat_c = np.zeros((7, 7))
for i in range(7):
    for j in range(7):
        mat_c[i, j] = fm.pearson_r(ann_all[rnames[i]], ann_all[rnames[j]])
im = ax.imshow(mat_c, cmap="RdYlGn", vmin=-1, vmax=1)
ax.set_xticks(range(7)); ax.set_xticklabels(rnames, fontsize=6, rotation=45)
ax.set_yticks(range(7)); ax.set_yticklabels(rnames, fontsize=6)
for i in range(7):
    for j in range(7):
        ax.text(j, i, f"{mat_c[i,j]:.2f}", ha="center", va="center", fontsize=6,
                color="black" if abs(mat_c[i,j]) > 0.3 else TEXT)
fig.colorbar(im, ax=ax, shrink=0.6, pad=0.02)
ax.set_title("7-River Cross-Correlation", fontsize=10, pad=6)

# 7-8,4-5: ONI + PDO time series
ax_oni = fig.add_subplot(gs[7, 4:6]); sty(ax_oni)
oni_yrs = sorted(oni.keys())
oni_vals = [oni[y] for y in oni_yrs]
colors_oni = [RED if v >= 0.5 else COOL_BLUE if v <= -0.5 else MUTED for v in oni_vals]
ax_oni.bar(oni_yrs, oni_vals, color=colors_oni, alpha=0.8, width=0.8)
ax_oni.axhline(0, color=MUTED, lw=0.5)
ax_oni.set_title("ONI (ENSO)", fontsize=9, pad=4)

ax_pdo = fig.add_subplot(gs[8, 4:6]); sty(ax_pdo)
pdo_yrs = sorted(pdo.keys())
pdo_vals = [pdo[y] for y in pdo_yrs]
colors_pdo = [RED if v > 0 else COOL_BLUE for v in pdo_vals]
ax_pdo.bar(pdo_yrs, pdo_vals, color=colors_pdo, alpha=0.8, width=0.8)
ax_pdo.axhline(0, color=MUTED, lw=0.5)
ax_pdo.set_title("PDO", fontsize=9, pad=4)

# 7-8,6-7: ENSO teleconnection scatter (Brazos + Willapa vs ONI)
ax = fig.add_subplot(gs[7, 6:]); sty(ax)
for obs_data, name, color in [(willapa_obs, "Willapa", CYAN), (tx["Brazos"], "Brazos", MAGENTA)]:
    clean = complete_years(obs_data)
    years = sorted(clean.keys())
    peak_by_year = {y: float(np.max(clean[y])) for y in years}
    metric, index = fm.align_index(peak_by_year, oni)
    ax.scatter(index, metric, s=12, c=color, alpha=0.5, edgecolors="none", label=name)
ax.set_xlabel("ONI", fontsize=7); ax.set_ylabel("peak flow (cfs)", fontsize=7)
ax.set_title("Peak Flow vs ONI", fontsize=9, pad=4)
ax.legend(fontsize=6, frameon=False)

ax = fig.add_subplot(gs[8, 6:]); sty(ax)
for obs_data, name, color in [(willapa_obs, "Willapa", CYAN), (tx["Brazos"], "Brazos", MAGENTA)]:
    clean = complete_years(obs_data)
    years = sorted(clean.keys())
    peak_by_year = {y: float(np.max(clean[y])) for y in years}
    metric, index = fm.align_index(peak_by_year, pdo)
    ax.scatter(index, metric, s=12, c=color, alpha=0.5, edgecolors="none", label=name)
ax.set_xlabel("PDO", fontsize=7); ax.set_ylabel("peak flow (cfs)", fontsize=7)
ax.set_title("Peak Flow vs PDO", fontsize=9, pad=4)
ax.legend(fontsize=6, frameon=False)

# ════════════════════════════════════════════════════════════════════════
# ROWS 9-10: RECORD BOOKS + TIMING + RETURN PERIOD + ANALOG
# ════════════════════════════════════════════════════════════════════════

# 9,0-1: Record Book (Willapa)
ax = fig.add_subplot(gs[9, 0:2]); sty(ax); ax.set_axis_off()
rb_w = fm.record_book(clean_w, n=5)
lines = ["  WILLAPA RECORD BOOK", ""]
lines.append("  Driest Summers")
for e in rb_w.driest_summers:
    lines.append(f"  #{e.rank}  {e.year}  {e.value:>8.1f} cfs  p{e.percentile*100:>3.0f}")
lines += ["", "  Wettest Years"]
for e in rb_w.wettest_years:
    lines.append(f"  #{e.rank}  {e.year}  {e.value:>8.1f} cfs  p{e.percentile*100:>3.0f}")
ax.text(0.05, 0.95, "\n".join(lines), transform=ax.transAxes, va="top",
        family="monospace", fontsize=7, color=TEXT)

# 9,2-3: Record Book (Brazos)
ax = fig.add_subplot(gs[9, 2:4]); sty(ax); ax.set_axis_off()
rb_b = fm.record_book(clean_b, n=5)
lines = ["  BRAZOS RECORD BOOK", ""]
lines.append("  Driest Summers")
for e in rb_b.driest_summers:
    lines.append(f"  #{e.rank}  {e.year}  {e.value:>8.1f} cfs  p{e.percentile*100:>3.0f}")
lines += ["", "  Wettest Years"]
for e in rb_b.wettest_years:
    lines.append(f"  #{e.rank}  {e.year}  {e.value:>8.1f} cfs  p{e.percentile*100:>3.0f}")
ax.text(0.05, 0.95, "\n".join(lines), transform=ax.transAxes, va="top",
        family="monospace", fontsize=7, color=TEXT)

# 9,4-5: Timing Drift (all rivers)
ax = fig.add_subplot(gs[9, 4:6]); sty(ax)
for name, obs_data in all_rivers.items():
    clean = complete_years(obs_data)
    if len(clean) < 10: continue
    tt = fm.center_of_timing_trend(clean)
    years = list(tt.years)
    ax.plot(years, tt.center_months, "-", color=RIVER_COLORS[name], lw=1, alpha=0.7, label=f"{name} {tt.days_per_decade:+.0f}d/dec")
ax.set_ylabel("center month", fontsize=7)
ax.set_title("Timing Drift — All Rivers", fontsize=9, pad=4)
ax.legend(fontsize=4, frameon=False, ncol=2)

# 9,6-7: Return Period (Willapa + Brazos)
ax = fig.add_subplot(gs[9, 6:]); sty(ax)
for obs_data, name, color in [(willapa_obs, "Willapa", CYAN), (tx["Brazos"], "Brazos", MAGENTA)]:
    years = sorted(obs_data.keys())
    peaks = np.array([float(np.nanmax(obs_data[y])) for y in years])
    n = len(peaks)
    sorted_p = np.sort(peaks)[::-1]
    rp = (n + 1) / np.arange(1, n + 1)
    ax.semilogy(sorted_p, rp, "o-", color=color, ms=2, lw=1, label=name)
ax.set_xlabel("peak flow (cfs)", fontsize=7); ax.set_ylabel("return period (yr)", fontsize=7)
ax.set_title("Return Period", fontsize=9, pad=4)
ax.legend(fontsize=6, frameon=False)
ax.grid(True, alpha=0.1, color=MUTED)

# --- Row 10: Flashiness, Cumulative, Analog, Percentile ---
# 10,0-1: Flashiness (all rivers)
ax = fig.add_subplot(gs[10, 0:2]); sty(ax)
for name, obs_data in all_rivers.items():
    clean = complete_years(obs_data)
    years = sorted(clean.keys())
    flash = [fm.flashiness(clean[y]) for y in years]
    ax.plot(years, flash, "-", color=RIVER_COLORS[name], lw=0.8, alpha=0.7, label=name)
ax.set_ylabel("flashiness", fontsize=7)
ax.set_title("Richards-Baker Flashiness", fontsize=9, pad=4)
ax.legend(fontsize=4, frameon=False, ncol=2)

# 10,2-3: Cumulative Flow (Willapa)
ax = fig.add_subplot(gs[10, 2:4]); sty(ax)
cmap_c = plt.get_cmap("viridis")
span_c = max(years_w) - min(years_w) or 1
for y in years_w:
    ax.plot(range(1, 13), np.nancumsum(willapa_obs[y]), color=cmap_c((y - min(years_w)) / span_c), lw=0.5, alpha=0.5)
mean_cum = np.nancumsum(np.nanmean([willapa_obs[y] for y in years_w], axis=0))
ax.plot(range(1, 13), mean_cum, color=AMBER, lw=2.5, label="mean")
ax.set_xticks(range(1, 13)); ax.set_xticklabels(MONTH_ABBR, fontsize=5)
ax.set_title("Willapa — Cumulative Flow", fontsize=9, pad=4)
ax.legend(fontsize=6, frameon=False)

# 10,4-5: Analog Years (Brazos)
ax = fig.add_subplot(gs[10, 4:6]); sty(ax)
target_b = max(clean_b.keys())
analogs_b = fm.analog_years(clean_b, target_b, n=8)
labels_b = [str(a.year) for a in analogs_b]
sims_b = [a.similarity for a in analogs_b]
colors_b = [CYAN if s > 0.9 else COOL_BLUE if s > 0.7 else MUTED for s in sims_b]
ax.barh(labels_b, sims_b, color=colors_b)
ax.invert_yaxis()
ax.set_xlabel("similarity (r)", fontsize=7)
ax.set_title(f"Brazos — Analog Years ({target_b})", fontsize=9, pad=4)

# 10,6-7: Annual KDE (Willapa + Brazos)
ax = fig.add_subplot(gs[10, 6:]); sty(ax)
for obs_data, name, color in [(willapa_obs, "Willapa", CYAN), (tx["Brazos"], "Brazos", MAGENTA)]:
    clean = complete_years(obs_data)
    years = sorted(clean.keys())
    annual = np.array([float(np.mean(clean[y])) for y in years])
    try:
        kde = gaussian_kde(annual)
        x_kde = np.linspace(annual.min() * 0.7, annual.max() * 1.3, 200)
        ax.plot(x_kde, kde(x_kde), color=color, lw=2, label=name)
        ax.fill_between(x_kde, kde(x_kde), color=color, alpha=0.1)
    except Exception: pass
ax.set_xlabel("annual mean (cfs)", fontsize=7)
ax.set_title("Annual Flow Distribution", fontsize=9, pad=4)
ax.legend(fontsize=6, frameon=False)

# ════════════════════════════════════════════════════════════════════════
# ROW 11: TX ANIMATION FRAMES + FOOTER
# ════════════════════════════════════════════════════════════════════════
if tx_frames and len(tx_frames) >= 6:
    picks = np.linspace(0, len(tx_frames) - 1, 6, dtype=int)
    for k, pick in enumerate(picks):
        ax = fig.add_subplot(gs[11, k])
        ax.set_facecolor(BG)
        for s in ax.spines.values(): s.set_visible(False)
        ax.imshow(plt.imread(tx_frames[pick]))
        ax.set_axis_off()

# Footer text in remaining space
ax = fig.add_subplot(gs[11, 6:]); ax.set_facecolor(BG); ax.set_axis_off()
ax.text(0.5, 0.5,
        "hydro-art\n\nUSGS NHDPlus HR\nUSGS NWIS\nNOAA CPC ONI\nNOAA NCEI PDO\n\nAll data federal\npublic domain",
        ha="center", va="center", fontsize=9, color=TICK, fontfamily="monospace",
        transform=ax.transAxes)

# ════════════════════════════════════════════════════════════════════════
# SAVE
# ════════════════════════════════════════════════════════════════════════
out_path = OUT / "master_poster.png"
fig.savefig(out_path, facecolor=BG, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out_path} ({out_path.stat().st_size / 1e6:.1f} MB)")


Saved: /Users/neilrunde/code/hydro-art/output/prints/master_poster.png (10.2 MB)


---
## Done

**Master poster saved to `output/prints/master_poster.png`.**

48x72" at 150 DPI with 40+ panels across Pacific County (WA) and Texas.
